# Labrador Minimal Downstream Ablation

This notebook is a compact downstream example for LabradorModel.

It does four things:
1. generates structured synthetic lab data,
2. creates binary labels from a nonlinear downstream rule,
3. trains LabradorModel as a classifier,
4. compares a small embedding-dimension sweep and a small learning-rate sweep.

## 1) Setup

This section imports the required libraries, defines a few constants, and sets the random seed so the example is reproducible.

In [2]:
import random

import numpy as np
import pandas as pd
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import LabradorModel


NUM_LABS = 10
LAB_CODES = [f"lab-{i}" for i in range(NUM_LABS)]
SEED = 42
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
DOWNSTREAM_SAMPLES = 600
EPOCHS = 5
BATCH_SIZE = 64


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/home/leemh/PyHealth/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.7.1+cu126
CUDA available: True


## 2) Synthetic Data

This section defines the synthetic downstream task used in the ablation.

- Samples are generated from latent clinical profiles with different lab co-occurrence patterns.
- Lab values depend on both code identity and profile-specific shifts.
- The downstream label follows the notebook rule: the target is positive when the interaction between `lab-2` and `lab-7` is large enough and `lab-1` stays below a threshold.

This produces a small nonlinear classification problem that is still fast to run.

In [3]:
def generate_structured_fake_batch(
    batch_size: int = 32,
    num_labs: int = NUM_LABS,
):
    """Generate synthetic lab bags with profile-specific code/value structure."""
    panel_probs = torch.tensor(
        [
            [0.05, 0.18, 0.07, 0.06, 0.05, 0.05, 0.18, 0.18, 0.10, 0.08],
            [0.12, 0.18, 0.16, 0.15, 0.06, 0.05, 0.13, 0.05, 0.05, 0.05],
            [0.10, 0.08, 0.05, 0.05, 0.16, 0.16, 0.10, 0.08, 0.12, 0.10],
        ],
        dtype=torch.float32,
    )

    anchor_codes = torch.tensor([1, 2, 7], dtype=torch.long)
    panel_ids = torch.randint(0, 3, (batch_size,))

    codes = torch.empty(batch_size, num_labs, dtype=torch.long)
    values = torch.empty(batch_size, num_labs, dtype=torch.float32)

    for i in range(batch_size):
        panel = panel_ids[i].item()
        extra_codes = torch.multinomial(
            panel_probs[panel],
            num_samples=num_labs - len(anchor_codes),
            replacement=True,
        )
        sample_codes = torch.cat([anchor_codes, extra_codes], dim=0)
        sample_codes = sample_codes[torch.randperm(num_labs)]

        sample_values = sample_codes.float() / (num_labs - 1)

        if panel == 0:
            sample_values[(sample_codes == 1) | (sample_codes == 6) | (sample_codes == 7)] += 0.18
            sample_values[sample_codes == 0] -= 0.05
        elif panel == 1:
            sample_values[(sample_codes == 2) | (sample_codes == 3)] += 0.15
            sample_values[sample_codes == 0] += 0.25
            sample_values[sample_codes == 1] += 0.05
            sample_values[sample_codes == 6] += 0.05
        else:
            sample_values[
                (sample_codes == 4)
                | (sample_codes == 5)
                | (sample_codes == 8)
                | (sample_codes == 9)
            ] += 0.18
            sample_values[sample_codes == 1] -= 0.06
            sample_values[sample_codes == 0] += 0.08

        sample_values += 0.05 * torch.randn_like(sample_values)
        sample_values = torch.clamp(sample_values, 0.0, 1.0)

        codes[i] = sample_codes
        values[i] = sample_values

    return codes, values


def get_mean_value_for_code(
    codes: torch.Tensor,
    values: torch.Tensor,
    target_code: int,
) -> torch.Tensor:
    mask = (codes == target_code).float()
    counts = mask.sum(dim=1)
    summed = (values * mask).sum(dim=1)
    return torch.where(
        counts > 0,
        summed / counts.clamp(min=1.0),
        torch.zeros_like(summed),
    )


def generate_downstream_labels(
    codes: torch.Tensor,
    values: torch.Tensor,
) -> torch.Tensor:
    code_1_value = get_mean_value_for_code(codes, values, target_code=1)
    code_2_value = get_mean_value_for_code(codes, values, target_code=2)
    code_7_value = get_mean_value_for_code(codes, values, target_code=7)
    return ((code_2_value * code_7_value > 0.20) & (code_1_value < 0.50)).long()


def make_downstream_samples(
    n_samples: int,
    seed: int,
    num_labs: int = NUM_LABS,
):
    set_seed(seed)
    codes, values = generate_structured_fake_batch(
        batch_size=n_samples,
        num_labs=num_labs,
    )
    labels = generate_downstream_labels(codes, values)

    samples = []
    for i in range(n_samples):
        samples.append(
            {
                "patient_id": f"patient-{i}",
                "visit_id": f"visit-{i}",
                "lab_codes": [LAB_CODES[int(code)] for code in codes[i].tolist()],
                "lab_values": values[i].tolist(),
                "label": int(labels[i].item()),
            }
        )
    return samples


def build_dataset(samples):
    return create_sample_dataset(
        samples=samples,
        input_schema={"lab_codes": "sequence", "lab_values": "tensor"},
        output_schema={"label": "binary"},
        dataset_name="labrador_downstream_demo",
    )


def split_samples(samples):
    n_samples = len(samples)
    train_end = int(n_samples * TRAIN_RATIO)
    val_end = int(n_samples * (TRAIN_RATIO + VAL_RATIO))
    return samples[:train_end], samples[train_end:val_end], samples[val_end:]

## 3) Training and Evaluation Helpers

This section defines the minimal training loop, evaluation function, and one helper that runs a single Labrador configuration on the downstream task.

In [4]:
def train_model(model, dataset, lr: float):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = get_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    for _ in range(EPOCHS):
        for batch in loader:
            optimizer.zero_grad()
            output = model(**batch)
            output["loss"].backward()
            optimizer.step()


def evaluate_model(model, dataset) -> float:
    model.eval()
    loader = get_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for batch in loader:
            output = model(**batch)
            y_true = output["y_true"].detach().cpu().view(-1).float()
            y_prob = output["y_prob"].detach().cpu()

            if y_prob.dim() == 2 and y_prob.shape[1] > 1:
                y_pred = torch.argmax(y_prob, dim=1).view(-1).float()
            else:
                y_pred = (y_prob.view(-1) > 0.5).float()

            y_true_all.append(y_true)
            y_pred_all.append(y_pred)

    y_true_all = torch.cat(y_true_all)
    y_pred_all = torch.cat(y_pred_all)
    return float((y_true_all == y_pred_all).float().mean().item())


def run_experiment(train_dataset, test_dataset, embed_dim: int, lr: float) -> float:
    set_seed(SEED)
    model = LabradorModel(
        dataset=train_dataset,
        code_feature_key="lab_codes",
        value_feature_key="lab_values",
        embed_dim=embed_dim,
        num_heads=2,
        num_layers=1,
    )
    train_model(model, train_dataset, lr=lr)
    return evaluate_model(model, test_dataset)

## 4) Create the Synthetic Downstream Dataset

This section generates one small synthetic dataset, shuffles it, splits it into train/validation/test subsets, and converts each split into a PyHealth dataset.

In [5]:
samples = make_downstream_samples(DOWNSTREAM_SAMPLES, seed=SEED)
random.Random(SEED).shuffle(samples)
train_samples, val_samples, test_samples = split_samples(samples)

train_dataset = build_dataset(train_samples)
val_dataset = build_dataset(val_samples)
test_dataset = build_dataset(test_samples)

print(f"train/val/test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"train positive rate: {np.mean([sample['label'] for sample in train_samples]):.3f}")
print(f"test positive rate: {np.mean([sample['label'] for sample in test_samples]):.3f}")

Label label vocab: {0: 0, 1: 1}
Label label vocab: {0: 0, 1: 1}
Label label vocab: {0: 0, 1: 1}
train/val/test: 360/120/120
train positive rate: 0.603
test positive rate: 0.600


## 5) Embedding Dimension Ablation

This section compares two hidden sizes while keeping the learning rate fixed at `1e-3`.

In [6]:
embed_dim_results = []
for embed_dim in [64, 128, 256]:
    accuracy = run_experiment(
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        embed_dim=embed_dim,
        lr=1e-3,
    )
    embed_dim_results.append({"embed_dim": embed_dim, "accuracy": accuracy})

embed_dim_df = pd.DataFrame(embed_dim_results)
embed_dim_df

/home/leemh/PyHealth/venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


,embed_dim,accuracy
0,64,0.466667
1,128,0.500000
2,256,0.425000


## 6) Learning Rate Ablation

This section compares two learning rates while keeping the model size fixed at `embed_dim=128`.

In [7]:
learning_rate_results = []
for lr in [1e-4, 1e-3, 5e-3]:
    accuracy = run_experiment(
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        embed_dim=128,
        lr=lr,
    )
    learning_rate_results.append({"learning_rate": lr, "accuracy": accuracy})

learning_rate_df = pd.DataFrame(learning_rate_results)
learning_rate_df

,learning_rate,accuracy
0,0.0001,0.608333
1,0.0010,0.500000
2,0.0050,0.425000


## 7) Summary

This section prints the final accuracy comparisons in a compact format.

In [8]:
print("Embedding Dimension Ablation")
for row in embed_dim_results:
    print(f"embed_dim={row['embed_dim']} -> accuracy={row['accuracy']:.4f}")
print()

print("Learning Rate Ablation")
for row in learning_rate_results:
    print(f"lr={row['learning_rate']:.0e} -> accuracy={row['accuracy']:.4f}")
print()

print("Embedding Dimension Results")
print(embed_dim_df.to_string(index=False))
print()

print("Learning Rate Results")
print(learning_rate_df.to_string(index=False))

Embedding Dimension Ablation
embed_dim=64 -> accuracy=0.4667
embed_dim=128 -> accuracy=0.5000
embed_dim=256 -> accuracy=0.4250

Learning Rate Ablation
lr=1e-04 -> accuracy=0.6083
lr=1e-03 -> accuracy=0.5000
lr=5e-03 -> accuracy=0.4250

Embedding Dimension Results
 embed_dim  accuracy
        64  0.466667
       128  0.500000
       256  0.425000

Learning Rate Results
 learning_rate  accuracy
        0.0001  0.608333
        0.0010  0.500000
        0.0050  0.425000
